In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames[:5]:
        print("   ", filename)

In [ ]:
DATA_DIR = "/kaggle/input/competitions/dog-breed-identification"

labels_path = os.path.join(DATA_DIR, "labels.csv")

labels = pd.read_csv(labels_path)

print("Dataset shape:", labels.shape)
print("\nFirst 5 rows:")
display(labels.head())

print("\nNumber of unique breeds:", labels["breed"].nunique())
print("\nNumber of images:", len(labels))

In [ ]:
breed_counts = labels["breed"].value_counts()

print("Number of breeds:", len(breed_counts))
print("\nImages per breed:")
display(breed_counts.head(10))

print("\nMinimum images in a breed:", breed_counts.min())
print("Maximum images in a breed:", breed_counts.max())

In [ ]:
breeds = sorted(labels["breed"].unique())

breed_to_index = {breed: i for i, breed in enumerate(breeds)}

labels["class"] = labels["breed"].map(breed_to_index)

print("Number of classes:", len(breeds))
print("\nFirst 10 class mappings:")

for breed, index in list(breed_to_index.items())[:10]:
    print(index, "->", breed)

In [ ]:
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

labels["image_path"] = labels["id"].apply(
    lambda x: os.path.join(TRAIN_DIR, x + ".jpg")
)

print(labels[["id", "breed", "class", "image_path"]].head())

In [ ]:
print("First image exists:", os.path.exists(labels["image_path"].iloc[0]))

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(12):
    img = tf.keras.utils.load_img(
        labels["image_path"].iloc[i],
        target_size=(160, 160)
    )

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(labels["breed"].iloc[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_df, val_df = train_test_split(
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels["class"]
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining classes:", train_df["class"].nunique())
print("Validation classes:", val_df["class"].nunique())

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label


train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["class"].values
    )
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["class"].values
    )
)

train_ds = (
    train_ds
    .shuffle(2000)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets created successfully.")

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

In [ ]:
NUM_CLASSES = 120


model = models.Sequential([
    
    layers.Input(shape=(160, 160, 3)),
    
    data_augmentation,

    layers.Conv2D(
        32, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.Conv2D(
        32, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.20),

    layers.Conv2D(
        64, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.Conv2D(
        64, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(
        128, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.Conv2D(
        128, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.Conv2D(
        256, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.Conv2D(
        256, 
        (3, 3), 
        padding="same", 
        activation="relu"
    ),
    layers.BatchNormalization(),

    layers.Dropout(0.35),

    layers.GlobalMaxPooling2D(),

    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.50),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
EPOCHS = 100

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
sample_submission_path = os.path.join(
    DATA_DIR,
    "sample_submission.csv"
)

sample_submission = pd.read_csv(sample_submission_path)

print("Sample submission shape:", sample_submission.shape)
print("\nColumns:")
print(sample_submission.columns.tolist()[:10])
print("...")

In [ ]:
test_ids = sample_submission["id"].values

test_paths = [
    os.path.join(TEST_DIR, image_id + ".jpg")
    for image_id in test_ids
]

test_ds = tf.data.Dataset.from_tensor_slices(test_paths)

def load_test_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image

test_ds = (
    test_ds
    .map(load_test_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Test images:", len(test_ids))

In [ ]:
predictions = model.predict(
    test_ds,
    verbose=1
)

print("Prediction shape:", predictions.shape)

In [ ]:
submission = pd.DataFrame(
    predictions,
    columns=sample_submission.columns[1:]
)

submission.insert(
    0,
    "id",
    sample_submission["id"]
)

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Submission file created!")
print("Location:", submission_path)
print("Shape:", submission.shape)

display(submission.head())

In [ ]:
print("File exists:", os.path.exists("/kaggle/working/submission.csv"))

print("\nSubmission shape:")
print(submission.shape)

print("\nFirst 5 rows:")
display(submission.head())

print("\nChecking for missing values:")
print(submission.isnull().sum().sum())

print("\nChecking prediction probabilities:")
print(
    submission.iloc[:, 1:].sum(axis=1).head()
)